# Main figures
Current submission graphics and frozen source values; 2026-10-09.
Production-event accounting for main balances; Figure 2 includes an explicitly conditional reference-building correction. Execute from project root or code directory.


In [ ]:
# Setup for current submission main figures
from pathlib import Path
import sys, json, math, os
if os.name=='nt':
    library_bin=Path(sys.executable).parent/'Library/bin'
    if library_bin.is_dir():
        os.environ['PATH']=str(library_bin)+os.pathsep+os.environ.get('PATH','')
        _dll_directory=os.add_dll_directory(str(library_bin))
import numpy as np
import pandas as pd
import matplotlib as mpl
mpl.use('Agg')
import matplotlib.pyplot as plt
from matplotlib.colors import FuncNorm,LinearSegmentedColormap,TwoSlopeNorm,ListedColormap
from matplotlib.patches import Patch,Rectangle
from matplotlib.lines import Line2D
from matplotlib.collections import LineCollection
from matplotlib.ticker import MaxNLocator
import cartopy,cartopy.crs as ccrs,cartopy.feature as cfeature
from PIL import Image,ImageChops
from lxml import etree
try: import pymupdf as fitz
except ImportError: import fitz
ROOT=Path.cwd() if (Path.cwd()/'code').exists() else Path.cwd().parent
sys.path.insert(0,str(ROOT/'code'))
from audit_panel_alignment import require_matplotlib_panel_alignment
S=ROOT/'output/Source_Data/CSV';Q=ROOT/'checks/main_figures';O=Q/'raw'
for p in [Q,O]:p.mkdir(parents=True,exist_ok=True)
cartopy.config['data_dir']=ROOT/'input/maps'
TABLES={}
NAME_MAP={'grid': 'Figure_1a', 'Figure1b': 'Figure_1b', 'Figure1c': 'Figure_1c', 'Figure1d': 'Figure_1d', 'components': 'Figure_2', 'interventions': 'Figure_4a', 'conditional_sensitivity': 'Figure_4b_values', 'ranking_comparisons': 'Figure_5b_summary', 'ranking_scenarios': 'Figure_5b'}
OFFSET_SCALE={'value_knots_percent': [0, 1, 2, 3, 5, 10, 20, 50, 100], 'colour_positions': [0, 0.08, 0.2, 0.35, 0.55, 0.72, 0.84, 0.94, 1], 'palette': ['#FFF2CC', '#F5C65D', '#DE9315', '#B2600B', '#763B08'], 'negative_colour': '#7040A0', 'definition': 'Piecewise-linear monotone display normalisation; raw ratios unchanged; all eligible values within range', 'reason': 'Eligible-cell median 2.955%, quartiles 2.420% and 4.869%; preserve nonuniform display and historical palette while matching the restored metric', 'quantiles': {'0.0': 1.0622036925456144, '0.1': 1.8116244254193934, '0.25': 2.419703723530511, '0.5': 2.955171176519898, '0.75': 4.868605417470285, '0.9': 6.173413747229923, '0.95': 10.942403427280539, '0.99': 25.447611379723075, '1.0': 90.43407108603581}, 'source': 'source_data/Figure1c.csv', 'source_sha256': '3859c4f0a6d203ae227baea0ba5c69fd0e3869906c1fbcb037dd9c15cfb409cb', 'setting_type': 'Author display setting, not a physical model parameter'}

def read(n):
    path=S/(NAME_MAP.get(n,n)+'.csv')
    df=pd.read_csv(path);TABLES[path.name]=df
    return df

def save_data(n,f):pass  # Published input CSVs remain frozen during rendering.
def show(n):pass


In [ ]:
mpl.rcParams.update({'font.family':'sans-serif','font.sans-serif':['DejaVu Sans','Arial'],'font.size':7,
 'axes.labelsize':7,'xtick.labelsize':6.5,'ytick.labelsize':6.5,'legend.fontsize':6,
 'axes.spines.top':False,'axes.spines.right':False,'axes.linewidth':.6,
 'svg.fonttype':'none','pdf.fonttype':42,'savefig.facecolor':'white','legend.frameon':False})
INK='#233F62';POS='#0072B2';NEG='#D55E00';GREY='#87949C'
COLORS={'Low':'#D89000','Reference':'#0072B2','High':'#AA3377'}
SCENARIO_MARKERS={'Low':'s','Reference':'o','High':'^'}
def name(s):return {'uk':'UK','usa':'USA','row':'Rest of world'}.get(s,s.replace('_',' ').title())
# read() is defined in Setup.
# Frozen input tables are not rewritten.
def show(n):pass
def panel(ax,l):
 ax.set_label(l)
 fig=ax.figure;fig.canvas.draw();renderer=fig.canvas.get_renderer()
 # Shared row anchors align a/b and c/d; shared column anchors align a/c and b/d.
 if l in ['b','d']:
  left=min(t.get_window_extent(renderer).x0 for target in [b,d] for t in target.get_yticklabels() if t.get_text())/fig.bbox.width
 else:left=a.get_position().x0
 row_top=(b if l in ['a','b'] else d).get_position().y1
 ax.annotate(l,(left,row_top),xycoords='figure fraction',xytext=(-6,9),textcoords='offset points',fontsize=9,fontweight='bold',color=INK,annotation_clip=False)

def export(fig,n,axes=None,**kwargs):
 fig.canvas.draw()
 if axes is not None:kwargs['exclude_axes']=[a for a in fig.axes if a not in axes]
 require_matplotlib_panel_alignment(fig,json_out=Q/f'Figure{n}_alignment.json',strict=True,**kwargs)
 fig.savefig(O/f'Figure{n}.pdf',dpi=600)
 fig.savefig(O/f'Figure{n}.svg',dpi=600)
 fig.savefig(O/f'Figure{n}.png',dpi=300)
 fig.savefig(O/f'Figure{n}.tiff',dpi=600,pil_kwargs={'compression':'tiff_lzw'})
 plt.close(fig);print('Exported Figure',n,flush=True)

# Figure 1: preserve the original two-map/two-bar composition.
g=read('grid');offsets=read('Figure1c');lon=np.arange(-180,181);lat=np.arange(-90,91)
def raster(frame,col):
 z=np.full((180,360),np.nan);ix=np.floor(frame.lon_center+180).astype(int);iy=np.floor(frame.lat_center+90).astype(int)
 z[iy,ix]=frame[col];return z
netmap=LinearSegmentedColormap.from_list('balance',[(0,'#8C2D04'),(.25,'#E66A12'),(.49,'#FEE6CE'),(.50,'#F7F7F7'),(.51,'#DCEAF4'),(.75,'#3B91BF'),(1,'#08446C')])
vmin=abs(g.net_gt.min()*1000);vmax=g.net_gt.max()*1000;soft=.05
def forward(x):
 x=np.asarray(x);return np.where(x<0,.5-.5*np.log1p(np.maximum(-x,0)/soft)/np.log1p(vmin/soft),.5+.5*np.log1p(np.maximum(x,0)/soft)/np.log1p(vmax/soft))
def inverse(y):
 y=np.asarray(y);return np.where(y<.5,-soft*np.expm1((1-2*y)*np.log1p(vmin/soft)),soft*np.expm1((2*y-1)*np.log1p(vmax/soft)))
fig=plt.figure(figsize=(7.2047244,135/25.4));gs=fig.add_gridspec(2,2,left=.02,right=.975,bottom=.14,top=.90,wspace=.70,hspace=.58,width_ratios=[1.65,1])
a=fig.add_subplot(gs[0,0],projection=ccrs.Robinson(central_longitude=10));b=fig.add_subplot(gs[0,1]);c=fig.add_subplot(gs[1,0],projection=ccrs.Robinson(central_longitude=10));d=fig.add_subplot(gs[1,1])
for ax in [a,c]:
 ax.add_feature(cfeature.LAND.with_scale('110m'),facecolor='#ECEEEF',edgecolor='none',zorder=1)
 ax.add_feature(cfeature.OCEAN.with_scale('110m'),facecolor='#F5F8FA',edgecolor='none',zorder=0)
 ax.add_feature(cfeature.COASTLINE.with_scale('110m'),edgecolor='#BCC3C6',linewidth=.25,zorder=4)
 ax.add_feature(cfeature.BORDERS.with_scale('110m'),edgecolor='#BCC3C6',linewidth=.15,zorder=4)
 ax.set_global();ax.set_xlim(-14172757.860816741,16060863.567199467)
 ax.set_ylim(ax.projection.transform_point(0,-55,ccrs.PlateCarree())[1],ax.projection.transform_point(0,75,ccrs.PlateCarree())[1])
 for lo,hi,bot,top in [(-25,33,43,71),(-63,-53,-28,-18)]:
  xx=np.r_[np.linspace(lo,hi,30),np.full(30,hi),np.linspace(hi,lo,30),np.full(30,lo)]
  yy=np.r_[np.full(30,bot),np.linspace(bot,top,30),np.full(30,top),np.linspace(top,bot,30)]
  ax.plot(xx,yy,color=INK,lw=.5,transform=ccrs.PlateCarree(),zorder=6)
mesh=a.pcolormesh(lon,lat,raster(g,'net_gt')*1000,cmap=netmap,norm=FuncNorm((forward,inverse),vmin=-vmin,vmax=vmax),transform=ccrs.PlateCarree(),rasterized=True,zorder=3)
cb=fig.colorbar(mesh,ax=a,fraction=.024,pad=.035);ticks=[v for v in [-10,-1,-.1,0,.1,1,10,100,500] if -vmin<=v<=vmax];cb.set_ticks(ticks);cb.set_ticklabels([f'{v:g}' for v in ticks]);cb.ax.tick_params(labelsize=5.5,length=2);cb.set_label('2025–2050 balance (Mt CO₂e)',fontsize=6)
# Restore the exact historical yellow/brown and purple palette.
scale=OFFSET_SCALE
knots=np.array(scale['value_knots_percent']);positions=np.array(scale['colour_positions'])
assert np.all(np.diff(knots)>0) and np.all(np.diff(positions)>0)
def ratio_forward(value):return np.interp(value,knots,positions)
def ratio_inverse(position):return np.interp(position,positions,knots)
offsetmap=LinearSegmentedColormap.from_list('offset',scale['palette'])
nonnegative=offsets[(offsets.net_gt>=0)&(offsets.gross_gt>0)];negative=offsets[offsets.net_gt<0]
assert len(nonnegative)+len(negative)==len(offsets)
mesh=c.pcolormesh(lon,lat,raster(nonnegative,'offset_pct'),cmap=offsetmap,norm=FuncNorm((ratio_forward,ratio_inverse),vmin=0,vmax=100),transform=ccrs.PlateCarree(),rasterized=True,zorder=3)
c.pcolormesh(lon,lat,np.where(np.isfinite(raster(negative,'net_gt')),1.,np.nan),cmap=ListedColormap([scale['negative_colour']]),vmin=0,vmax=1,transform=ccrs.PlateCarree(),rasterized=True,zorder=3.5)
cb=fig.colorbar(mesh,ax=c,fraction=.024,pad=.035);cb.set_ticks([0,2,5,10,20,50,100]);cb.ax.tick_params(labelsize=5.5,length=2);cb.set_label('Gross savings offset (%)',fontsize=6)
c.legend(handles=[Patch(facecolor=scale['negative_colour'],label='Net-emitting cell')],loc='upper center',bbox_to_anchor=(.52,-.035),fontsize=6,handlelength=1.1)

tot=g.groupby('country_tag').net_gt.sum().sort_values();neg=-tot[tot<0]*1000
b.barh(range(len(neg)),neg,color=NEG,height=.57)
b.set(yticks=range(len(neg)),yticklabels=[name(x) for x in neg.index],ylim=(len(neg)-.4,-.6),xlim=(0,neg.max()*1.23),xlabel='Cumulative net emissions\n(Mt CO₂e, 2025–2050)')
for j,v in enumerate(neg):b.text(v+2,j,f'{v:.1f}',va='center',fontsize=6)
zones=read('Figure1d').set_index('climate_zone')
short={'humid_continental_temperate':'Humid temperate','humid_subtropical_dense':'Humid subtropical','tropical_wet_dense':'Tropical wet','hot_arid_sparse':'Hot arid sparse','hot_arid_dense_riverine':'Hot arid riverine','hot_arid_low_density':'Hot arid low density'}
v=zones.offset_pct
span=max(v.max(),0)-min(v.min(),0);pad=max(span*.04,1)
d.barh(range(6),v,color='#D89000',height=.57)
d.axvline(0,color='#87949C',lw=.6,zorder=0)
d.set(yticks=range(6),yticklabels=[short[z] for z in zones.index],ylim=(5.6,-.6),xlim=(0,v.max()*1.5),xlabel='Median gross savings offset (%)')
for j,value in enumerate(v):
 d.text(value+.12 if value>=0 else value-.12,j,f'{value:.1f}',va='center',ha='left' if value>=0 else 'right',fontsize=6)
for ax,l in zip([a,b,c,d],'abcd'):panel(ax,l)
# Time windows and event-accounting scope are defined in the caption.
export(fig,1,[a,b,c,d],exemptions=[{'panels':['a','c'],'checks':['row','panel-label'],'reason':'Robinson maps retain fixed geographic aspect alongside full-height horizontal bar panels.'}])
# Source summaries are created by summarize.py; existing Figure1b is preserved.



In [ ]:
mpl.rcParams.update({'font.family':'sans-serif','font.sans-serif':['DejaVu Sans','Arial'],'font.size':7,
    'axes.labelsize':7,'xtick.labelsize':6,'ytick.labelsize':6,'legend.fontsize':6,
    'axes.spines.top':False,'axes.spines.right':False,'axes.linewidth':.6,'pdf.fonttype':42,'svg.fonttype':'none','legend.frameon':False})
GREEN='#009E73';BLUE='#0072B2';ORANGE='#D55E00';INK='#171717'

def draw_figure2(data_dir,out,checks):
    values=read('components')
    ranges=values[['country','storage_thermal_lower_g_kwh','storage_thermal_upper_g_kwh','no_storage_thermal_lower_g_kwh','no_storage_thermal_upper_g_kwh']].copy()
    sr=values[['thermal_g_CO2e_kwh','scenario_increment_net_g_kwh']].copy()
    nr=values[['no_storage_thermal_g_CO2e_kwh','no_storage_scenario_increment_net_g_kwh']].rename(columns=lambda c:c.removeprefix('no_storage_'))
    assert np.allclose(sr.thermal_g_CO2e_kwh,nr.thermal_g_CO2e_kwh,atol=1e-9)
    n=len(values);assert len(ranges)==n and n>0
    comps=[('displaced_grid','Grid credit',GREEN),('pv_embodied','PV system emissions','#B8BDC5'),('battery','Battery emissions',BLUE)]
    assert np.allclose(values[[k+'_g_kwh' for k,_,_ in comps]].sum(axis=1),values.net_g_kwh,atol=1e-9)
    values['thermal_g_kwh']=sr.thermal_g_CO2e_kwh
    values['net_g_kwh']=sr.scenario_increment_net_g_kwh
    values['no_storage_g_kwh']=nr.scenario_increment_net_g_kwh
    split=values.copy()
    values['building_extra_use_g_kwh']=split.building_extra_use_g_kwh
    values['building_savings_g_kwh']=split.building_savings_g_kwh
    assert np.all(values.building_extra_use_g_kwh<=0) and np.all(values.building_savings_g_kwh>=0)
    assert np.allclose(values.building_extra_use_g_kwh+values.building_savings_g_kwh,values.thermal_g_kwh,atol=1e-9)
    comps.extend([('building_extra_use','Extra building use',ORANGE),('building_savings','Building energy savings','#E69F00')])
    assert np.allclose(ranges.storage_thermal_lower_g_kwh,ranges.no_storage_thermal_lower_g_kwh,atol=1e-9)
    assert np.allclose(ranges.storage_thermal_upper_g_kwh,ranges.no_storage_thermal_upper_g_kwh,atol=1e-9)
    fig=plt.figure(figsize=(7.2047244,4.4))
    a=fig.add_axes([.18,.18,.78,.61])
    pos=np.zeros(n);neg=np.zeros(n)
    for k,title,col in comps:
        v=values[k+'_g_kwh'].to_numpy()
        a.barh(range(n),v,left=np.where(v>=0,pos,neg),color=col,height=.47,edgecolor='white',linewidth=.25)
        pos+=np.maximum(v,0);neg+=np.minimum(v,0)
    for j,row in values.iterrows():
        width=-row.battery_g_kwh-row.building_extra_use_g_kwh;right=row.pv_embodied_g_kwh
        if width>0:a.add_patch(Rectangle((right-width,j-.235),width,.47,facecolor='none',edgecolor='black',lw=.8,zorder=5))
        a.plot([row.net_g_kwh,row.no_storage_g_kwh],[j,j],color='white',lw=2,zorder=6)
        a.plot([row.net_g_kwh,row.no_storage_g_kwh],[j,j],color=INK,lw=.65,zorder=7)
    a.scatter(values.no_storage_g_kwh,range(n),marker='o',s=22,facecolors='white',edgecolors=INK,lw=.8,zorder=10)
    a.scatter(values.net_g_kwh,range(n),marker='D',s=22,c=INK,edgecolors='white',lw=.65,zorder=10)
    a.axvline(0,c='#87949C',lw=.6)
    names=[x.replace('_',' ').title() for x in values.country]
    a.set(yticks=range(n),yticklabels=names,ylim=(n-.4,-.6),
        xlim=(np.floor(neg.min()*1.12/10)*10,np.ceil(pos.max()*1.12/10)*10),
        xlabel='Incremental ten-year contribution\n(g CO₂e per added generated kWh; 2040 additions)')
    handles=[Patch(color=col,label=title) for _,title,col in comps]+[
        Line2D([],[],ls='none',marker='o',mfc='white',mec=INK,ms=4,label='PV only'),
        Line2D([],[],ls='none',marker='D',mfc=INK,mec='white',ms=4,label='PV + storage'),
        Patch(facecolor='none',edgecolor='black',label='Additional emissions')]
    # Two aligned rows; Matplotlib fills legend entries down each column.
    order=[0,4,1,5,2,6,3,7]
    gap_pt=6
    a.legend(handles=[handles[i] for i in order],loc='lower center',
        bbox_to_anchor=(.5,1+gap_pt/(fig.get_figheight()*72*a.get_position().height)),
        borderaxespad=0,borderpad=0,ncol=4,fontsize=6,columnspacing=1.4,
        labelspacing=.65,handlelength=1.1)
    a.set_label('main')
    out.mkdir(parents=True,exist_ok=True);checks.mkdir(parents=True,exist_ok=True)
    fig.canvas.draw()
    require_matplotlib_panel_alignment(fig,axes=[a],panel_ids=['main'],json_out=checks/'Figure2_alignment.json',strict=True)
    fig.savefig(out/'Figure2.pdf',dpi=600)
    fig.savefig(out/'Figure2.svg',dpi=600)
    fig.savefig(out/'Figure2.png',dpi=300)
    fig.savefig(out/'Figure2.tiff',dpi=600,pil_kwargs={'compression':'tiff_lzw'})
    plt.close(fig)

draw_figure2(S,O,Q)


In [ ]:
plt.rcParams.update({'font.family': 'sans-serif', 'font.sans-serif': ['DejaVu Sans', 'Arial'],
    'font.size': 7, 'axes.labelsize': 7, 'axes.titlesize': 8, 'xtick.labelsize': 6.5,
    'ytick.labelsize': 6.5, 'legend.fontsize': 6.5, 'axes.linewidth': .6,
    'lines.linewidth': 1.4, 'pdf.fonttype': 42, 'svg.fonttype': 'none',
    'savefig.facecolor': 'white', 'axes.spines.top': False, 'axes.spines.right': False})
BLUE, RED, GOLD, TEAL, GREY = '#267b9b', '#bd5145', '#d9a441', '#378d75', '#6d747d'
a = pd.read_csv(S/'Figure_3a.csv')
dec = pd.read_csv(S/'Figure_3b.csv')
env = pd.read_csv(S/'Figure_3a.csv')
fig, axes = plt.subplots(2, 3, figsize=(7.2047244, 6.2204724))  # 183 x 158 mm
fig.subplots_adjust(left=.10, right=.97, top=.79, bottom=.13, wspace=.38, hspace=.55)


def panel(ax, letter, title):
    ax.set_label(letter)
    ax.set_title(title, loc='left', pad=9)
    ax.annotate(letter, (0, 1), xycoords='axes fraction', xytext=(-18, 9),
                textcoords='offset points', fontsize=9, fontweight='bold', annotation_clip=False)


selected_rows = []
for j, country in enumerate(['paraguay', 'switzerland', 'france']):
    ax = axes[0, j]
    for scenario, color in [('s1_d0', RED), ('stop2035', TEAL)]:
        q = env.query('country == @country and scenario == @scenario').sort_values('year')
        ax.fill_between(q.year.to_numpy(), q.cumulative_lower_mt.to_numpy(), q.cumulative_upper_mt.to_numpy(),
                        facecolor=color, edgecolor='none', alpha=.18, zorder=1)
        selected_rows.append(q)
    for scenario, field, color, ls, label in [
            ('s1_d0', 'service_net_gt', GREY, '--', 'Service allocation'),
            ('s1_d0', 'net_gt', RED, '-', 'Production events'),
            ('stop2035', 'net_gt', TEAL, '-', 'Stop additions after 2035')]:
        q = a.query('country == @country and scenario == @scenario').sort_values('year')
        ax.plot(q.year, q[field].cumsum() * 1000, color=color, ls=ls, label=label, zorder=3)
    ax.axhline(0, color='black', lw=.6)
    ax.set_xticks([2025, 2035, 2050]); ax.set_xlabel('Year')
    if j == 0:
        ax.set_ylabel('Cumulative mitigation (Mt)')
    panel(ax, 'abc'[j], country.capitalize())
    ax = axes[1, j]
    q = dec.query('country == @country').sort_values('year')
    assert len(q) == 26
    np.testing.assert_allclose(q.operating_mt + q.new_build_mt + q.replacement_mt, q.net_mt, atol=1e-7)
    ax.bar(q.year, q.operating_mt, color=BLUE, width=.85, label='Operating balance', zorder=1)
    ax.bar(q.year, q.new_build_mt, color=RED, width=.85, label='New capacity production', zorder=1)
    ax.bar(q.year, q.replacement_mt, bottom=q.new_build_mt, color=GOLD, width=.85, label='Component replacements', zorder=1)
    bound = env.query('country == @country and scenario == "s1_d0"').sort_values('year')
    ax.fill_between(bound.year.to_numpy(), bound.annual_lower_mt.to_numpy(), bound.annual_upper_mt.to_numpy(),
                    facecolor='#454545', alpha=.20, edgecolor='none', zorder=2)
    ax.plot(q.year, q.net_mt, color='black', lw=1, label='Annual net', zorder=3)
    ax.axhline(0, color='black', lw=.6)
    ax.set_xticks([2025, 2035, 2050]); ax.set_xlabel('Year')
    if j == 0:
        ax.set_ylabel('Annual carbon balance (Mt)')
    panel(ax, 'def'[j], '')

# Two single-row legends, measured from the final country-title boundary.
fig.canvas.draw()
renderer=fig.canvas.get_renderer()
title_top=max(ax.title.get_window_extent(renderer).y1 for ax in axes[0])
title_top=max(ax._left_title.get_window_extent(renderer).y1 for ax in axes[0])
center=(axes[0,0].get_position().x0+axes[0,2].get_position().x1)/2
handles, labels = axes[1, 0].get_legend_handles_labels()
annual_legend=fig.legend(handles, labels, ncol=4, loc='lower center',
    bbox_to_anchor=(center,(title_top+8*fig.dpi/72)/fig.bbox.height),
    frameon=False,borderaxespad=0,borderpad=0,columnspacing=1.6,handlelength=2)
fig.canvas.draw()
annual_top=annual_legend.get_window_extent(renderer).y1
handles, labels = axes[0, 0].get_legend_handles_labels()
fig.legend(handles, labels, ncol=3, loc='lower center',
    bbox_to_anchor=(center,(annual_top+4*fig.dpi/72)/fig.bbox.height),
    frameon=False,borderaxespad=0,borderpad=0,columnspacing=1.8,handlelength=2)
# Inventory-band definition is retained in the caption.
# Source and boundary explanations are retained in the caption.
fig.canvas.draw()
require_matplotlib_panel_alignment(fig, json_out=Q/'Figure3_alignment.json', strict=True)
fig.savefig(O / 'Figure3.pdf')
fig.savefig(O / 'Figure3.svg')
fig.savefig(O / 'Figure3.png', dpi=300)
fig.savefig(O / 'Figure3.tiff', dpi=600, pil_kwargs={'compression': 'tiff_lzw'})
plt.close(fig)
# Source envelope columns are already in the frozen CSV.
print('Figure 3 exported; alignment passed; core central data updated.')


In [ ]:
mpl.rcParams.update({'font.family':'sans-serif','font.sans-serif':['DejaVu Sans','Arial'],'font.size':7,
 'axes.labelsize':7,'xtick.labelsize':6.5,'ytick.labelsize':6.5,'legend.fontsize':6,
 'axes.spines.top':False,'axes.spines.right':False,'axes.linewidth':.6,
 'svg.fonttype':'none','pdf.fonttype':42,'savefig.facecolor':'white','legend.frameon':False})
INK='#233F62';POS='#0072B2';NEG='#D55E00';GREY='#87949C'
COLORS={'Low':'#D89000','Reference':'#0072B2','High':'#AA3377'}
SCENARIO_MARKERS={'Low':'s','Reference':'o','High':'^'}
def name(s):return {'uk':'UK','usa':'USA','row':'Rest of world'}.get(s,s.replace('_',' ').title())
# read() is defined in Setup.
def save_data(n,f):pass
def show(n):pass
def panel(ax,l):
 ax.set_label(l);ax.annotate(l,(0,1),xycoords='axes fraction',xytext=(-6,9),textcoords='offset points',fontsize=9,fontweight='bold',color=INK,annotation_clip=False)
def export(fig,n,axes=None,**kwargs):
 fig.canvas.draw()
 if axes is not None:kwargs['exclude_axes']=[a for a in fig.axes if a not in axes]
 require_matplotlib_panel_alignment(fig,json_out=Q/f'Figure{n}_alignment.json',strict=True,**kwargs)
 fig.savefig(O/f'Figure{n}.pdf',dpi=600)
 fig.savefig(O/f'Figure{n}.svg',dpi=600)
 fig.savefig(O/f'Figure{n}.png',dpi=300)
 fig.savefig(O/f'Figure{n}.tiff',dpi=600,pil_kwargs={'compression':'tiff_lzw'})
 plt.close(fig);print('Exported Figure',n,flush=True)

# Figure 4: original intervention bars and conditional violins.
lever_all=read('interventions');u=read('conditional_sensitivity')
lever=lever_all.copy()
assert len(lever)==9 and lever.intervention.is_unique
tags=read('Figure1b').country_tag.tolist()
baseline=read('components').set_index('country');rows=[]
for country in tags:
 v=u.query('country==@country').net_g_kwh.to_numpy();assert len(v)==192
 q=np.quantile(v,[.25,.5,.75]);rows.append(dict(country=country,p25=q[0],median=q[1],p75=q[2],baseline=baseline.loc[country,'net_g_kwh'],negative_scenarios_pct=100*(v<0).mean()))
summary=pd.DataFrame(rows)
fig,(a,b)=plt.subplots(1,2,figsize=(7.2047244,4.8));fig.subplots_adjust(left=.22,right=.925,bottom=.20,top=.83,wspace=.67)
labels={'Manufacturing procurement': 'Wind electricity\nprocurement', 'Direct use +20 pp': 'Direct use', 'Half-size storage': 'Half battery capacity', 'Half storage + direct use': 'Half battery capacity\n+ direct use', 'Half storage + procurement': 'Half battery capacity\n+ wind procurement', 'All three changes': 'Three changes combined', 'PV without storage': 'No storage', 'Smelter power P3': 'Smelter electricity\nchange', 'Smelter P3 + three changes': 'Three changes +\nsmelter electricity'}
for j,row in enumerate(lever.itertuples()):
 a.barh(j,row.reduction_pct,height=.5,color='#707780' if row.intervention=='PV without storage' else POS,hatch='///' if row.intervention=='PV without storage' else None,edgecolor='white')
 a.text(row.reduction_pct+1,j,f'{row.reduction_pct:.1f}',va='center',fontsize=6)
a.set(yticks=range(len(lever)),yticklabels=[labels[v] for v in lever.intervention],ylim=(len(lever)-.4,-.6),xlim=(0,100),xticks=[0,25,50,75,100],xlabel='Reduction in remaining\nnet emissions (%)');a.spines['left'].set_visible(False);a.tick_params(axis='y',length=0)
for j,row in enumerate(summary.itertuples()):
 v=u.query('country==@row.country').net_g_kwh.to_numpy();vi=b.violinplot([v],positions=[j],orientation='horizontal',widths=.7,showextrema=False)
 for body in vi['bodies']:body.set_facecolor('#C6DEEF');body.set_edgecolor(POS);body.set_linewidth(.65);body.set_alpha(1)
 b.plot([row.p25,row.p75],[j,j],c=POS,lw=3);b.scatter(row.median,j,s=17,c='white',edgecolors=POS,lw=.7,zorder=4);b.scatter(row.baseline,j,marker='D',s=20,c='#171717',edgecolors='white',lw=.6,zorder=5)
b.set(yticks=range(len(tags)),yticklabels=[name(t) for t in tags],ylim=(len(tags)-.4,-.6),xlabel='Incremental net balance\n(g CO₂e per added generated kWh)');b.axvline(0,c=GREY,ls=':',lw=.65)
for ax,l in zip([a,b],'ab'):panel(ax,l)
# Conditional-scenario definition is retained in the caption.
export(fig,4)




In [ ]:
mpl.rcParams.update({'font.family':'sans-serif','font.sans-serif':['DejaVu Sans','Arial'],'font.size':7,
 'axes.labelsize':7,'xtick.labelsize':6.5,'ytick.labelsize':6.5,'legend.fontsize':6,
 'axes.spines.top':False,'axes.spines.right':False,'axes.linewidth':.6,
 'svg.fonttype':'none','pdf.fonttype':42,'savefig.facecolor':'white','legend.frameon':False})
INK='#233F62'; POS='#0072B2'; NEG='#D55E00'; GREY='#87949C'
COLORS={'Low':'#D89000','Reference':'#0072B2','High':'#AA3377'}
SCENARIO_MARKERS={'Low':'s','Reference':'o','High':'^'}
ORDER=['Uniform','Screened','Yield per roof','Procurement','Combined']
DISPLAY={'Uniform': 'Uniform', 'Screened': 'Screened', 'Yield per roof': 'Yield per roof', 'Procurement': 'Wind electricity\nprocurement', 'Combined': 'Screening +\nwind procurement'}
STYLE={'Uniform':('#0072B2','o',False),'Screened':('#171717','D',True),
       'Yield per roof':('#171717','^',False),'Procurement':('#009E73','D',False),
       'Combined':('#009E73','P',True)}
used=[]


def read(name):
    p=S/(NAME_MAP.get(name,name)+'.csv'); used.append(p)
    return pd.read_csv(p)


def panel(ax, letter):
    ax.set_label(letter)
    ax.annotate(letter,(0,1),xycoords='axes fraction',xytext=(-6,9),textcoords='offset points',
                fontsize=9,fontweight='bold',color=INK,annotation_clip=False)


abc="paired=read('Figure_5a_values').set_index('sample_id').sort_index()\nassert len(paired)==2048 and set(paired.columns)=={'Screened','Procurement','Combined'}\ncats=[('Screened','Reference','Screened','#171717'),('Procurement','Procurement','Uniform','#009E73'),('Combined','Procurement','Screened','#009E73')]\nfig,axes=plt.subplots(2,2,figsize=(7.2047244,6.0));fig.subplots_adjust(left=.115,right=.98,bottom=.14,top=.90,hspace=.34,wspace=.48);a,b,c,d=axes.flat\nsegments=np.stack([np.tile(np.arange(len(cats)),(len(paired),1)),paired.to_numpy()],axis=2);a.add_collection(LineCollection(segments,colors=GREY,alpha=.006,linewidths=.25,rasterized=True))\nqrows=[]\nfor j,(lab,sc,st,col) in enumerate(cats):\n    q=np.quantile(paired[lab],[.05,.5,.95])\n    a.errorbar(j,q[1],yerr=[[q[1]-q[0]],[q[2]-q[1]]],fmt=['D','D','P'][j],color=col,ms=5,mfc='white' if lab=='Procurement' else col,mec=col if lab=='Procurement' else 'white',mew=.45,capsize=3,elinewidth=1.5,zorder=4)\n    qrows.append(dict(category=lab,p05_gt=q[0],median_gt=q[1],p95_gt=q[2]))\na.autoscale_view();a.axhline(0,c=GREY,lw=.6,ls=':');a.set(xlim=(-.45,len(cats)-.45),xticks=range(len(cats)),xticklabels=[DISPLAY.get(r[0],r[0]) for r in cats],ylabel='Gain relative to uniform deployment (Gt CO₂e)');a.tick_params(axis='x',labelsize=5.7)\nrank=read('ranking_comparisons');rank=rank[rank.rule.isin(['2025 grid','Yield per roof','Total generation'])].reset_index(drop=True)\nranking_draws=read('ranking_scenarios').query('sample_id>0');ranking_summary=[]\nfor j,row in enumerate(rank.itertuples()):\n    z=ranking_draws.query('rule==@row.rule');v=z.full_minus_alternative_gt;assert len(v)==2048\n    jitter=((z.sample_id.to_numpy()*37)%101/100-.5)*.27\n    b.scatter(v,j+jitter,s=3,c=GREY,alpha=.07,edgecolors='none',rasterized=True)\n    q=np.quantile(v,[.05,.5,.95]);tie=np.max(abs(v))<1e-8\n    b.errorbar(q[1],j,xerr=[[q[1]-q[0]],[q[2]-q[1]]],fmt='o',color=INK if tie else '#0072B2',mfc='white' if tie else '#0072B2',ms=5,capsize=3,elinewidth=1.5,zorder=4)\n    ranking_summary.append(dict(rule=row.rule,retained_regions=row.retained_regions,p05_gt=q[0],median_gt=q[1],p95_gt=q[2]))\nb.axvline(0,c=GREY,lw=.65,ls=':');b.set(yticks=range(3),yticklabels=[r.rule for r in rank.itertuples()],ylim=(2.65,-.65),xlabel='Screened − alternative (Gt CO₂e)');b.tick_params(axis='y',length=0,labelsize=5.8);b.spines['left'].set_visible(False)\nbins=np.linspace(.5,1.5,9);binrows=[];pointrows=[]\nfor idx,sc in enumerate(['Low','Reference','High']):\n    source=read('Figure_5c_values').query('scenario==@sc').set_index('sample_id').sort_index()\n    gain=source.screening_gain_gt;ef=source.displaced_ef_multiplier\n    c.scatter(ef,gain,s=3,c=COLORS[sc],alpha=.05,edgecolors='none',rasterized=True)\n    pointrows.extend(dict(sample_id=int(sid),scenario=sc,displaced_ef_multiplier=float(xx),screening_gain_gt=float(yy)) for sid,xx,yy in zip(gain.index,ef,gain))\n    rr=[]\n    for k,(lo,hi) in enumerate(zip(bins[:-1],bins[1:])):\n        mask=(ef>=lo)&(ef<hi if k<7 else ef<=hi);q=np.quantile(gain[mask],[.05,.5,.95]);rr.append(dict(scenario=sc,bin_center=(lo+hi)/2,plot_x=(lo+hi)/2+(idx-1)*.014,p05_gt=q[0],median_gt=q[1],p95_gt=q[2]))\n    ff=pd.DataFrame(rr);binrows.extend(rr);c.errorbar(ff.plot_x,ff.median_gt,yerr=[ff.median_gt-ff.p05_gt,ff.p95_gt-ff.median_gt],fmt=['s-','o-','^-'][idx],c=COLORS[sc],ms=4,mfc='white',mec=COLORS[sc],mew=.75,lw=1,capsize=1.4,elinewidth=.75,label=sc)\nc.axhline(0,c=GREY,lw=.6,ls=':');c.set(xlim=(.5,1.5),xticks=[.5,.75,1,1.25,1.5],xlabel='Grid emission factor multiplier',ylabel='Screened − uniform (Gt CO₂e)')\n"
raw=read('Figure_5d_values')
assert len(raw)==2048*5 and not raw.duplicated(['sample_id','strategy']).any()
assert set(raw.strategy)==set(ORDER)
assert all(set(g.sample_id)==set(range(1,2049)) for _,g in raw.groupby('strategy'))
summary=[]
for strategy in ORDER:
    g=raw.query('strategy==@strategy')
    row={'strategy':strategy,'n_draws':len(g)}
    for field, prefix in [('gain_gt','gain'),('national_net_emissions_mt','emissions')]:
        for p, value in zip([5,25,50,75,95],np.quantile(g[field],[.05,.25,.5,.75,.95])):
            row[f'{prefix}_p{p:02d}']=value
    summary.append(row)
summary=pd.DataFrame(summary).set_index('strategy').loc[ORDER]
old=read('Figure_5d').set_index('strategy').loc[ORDER]
for prefix, oldprefix, unit in [('gain','gain','gt'),('emissions','emissions','mt')]:
    for q, oq in [('p05','p05'),('p50','median'),('p95','p95')]:
        np.testing.assert_allclose(summary[f'{prefix}_{q}'],old[f'{oldprefix}_{oq}_{unit}'],rtol=1e-12,atol=1e-12)
# Summary is verified against the frozen source table.


def original_scatter(d):
    # Reuse the original Figure5d scatter/errorbar grammar with current draws.
    d.scatter(raw.gain_gt,raw.national_net_emissions_mt,s=3,c='#929292',
              alpha=.10,edgecolors='none',rasterized=True,zorder=1)
    for strategy in ORDER:
        r=summary.loc[strategy];col,marker,filled=STYLE[strategy]
        d.errorbar(r.gain_p50,r.emissions_p50,
            xerr=[[r.gain_p50-r.gain_p05],[r.gain_p95-r.gain_p50]],
            yerr=[[r.emissions_p50-r.emissions_p05],[r.emissions_p95-r.emissions_p50]],
            fmt=marker,c=col,mfc=col if filled else 'white',mec=col,ms=4.5,mew=.7,
            elinewidth=.8,capsize=2,zorder=3)
    d.axhline(0,c=GREY,lw=.6,ls=':')
    d.set(xlim=(min(-.1,float(raw.gain_gt.min())*1.06),float(raw.gain_gt.max())*1.06),
          ylim=(-max(5,float(raw.national_net_emissions_mt.max())*.025),float(raw.national_net_emissions_mt.max())*1.08),
          xlabel='Gain relative to uniform deployment (Gt CO₂e)',
          ylabel='Residual net emissions (Mt CO₂e)')
    d.legend(handles=[Line2D([],[],color=STYLE[s][0],marker=STYLE[s][1],ls='none',
        mfc=STYLE[s][0] if STYLE[s][2] else 'white',mec=STYLE[s][0],ms=4.5,
        label=DISPLAY[s]) for s in ORDER],loc='upper right',bbox_to_anchor=(.98,.98),
        borderaxespad=0,ncol=2,fontsize=5.7,columnspacing=.9,handletextpad=.4,labelspacing=.6)
    panel(d,'d')


position_checks=[]
for variant in ['originalstyle']:
    # Exact reuse of approved a-c source. No regeneration of model outputs.
    exec(compile(abc,'<Figure5 panels a-c>','exec'),globals())
    for ax,letter in zip([a,b,c],'abc'):
        panel(ax,letter)
    c.legend(handles=[Line2D([],[],color=COLORS[s],lw=1.3,marker=SCENARIO_MARKERS[s],mfc='white',ms=4,label=s)
        for s in ['Low','Reference','High']],loc='upper left',bbox_to_anchor=(.035,.98),borderaxespad=0,
        ncol=1,fontsize=6.5,labelspacing=.6,handlelength=1.8,handletextpad=.6)
    c.get_legend().set_title('Manufacturing electricity',prop={'size':6.5})
    positions={letter:ax.get_position().bounds for letter,ax in zip('abc',[a,b,c])}
    if position_checks:
        for letter in 'abc':
            np.testing.assert_array_equal(positions[letter],position_checks[0][letter])
    position_checks.append(positions)
    original_scatter(d)
    alignment=dict(panel_ids={a:'a',b:'b',c:'c',d:'d'},
                   row_groups=[['a','b'],['c','d']],column_groups=[['a','c'],['b','d']])
    fig.canvas.draw()
    require_matplotlib_panel_alignment(fig,json_out=Q/'Figure5_alignment.json',strict=True,**alignment)
    stem=O/'Figure5'
    fig.savefig(str(stem)+'.pdf',dpi=600)
    fig.savefig(str(stem)+'.svg',dpi=600)
    fig.savefig(str(stem)+'.png',dpi=300)
    fig.savefig(str(stem)+'.tiff',dpi=600,pil_kwargs={'compression':'tiff_lzw'})
    plt.close(fig)
print('Figure 5 exported from all 2,048 paired draws per strategy.')


In [ ]:
# Source data and final exports
def sha(p):return __import__("hashlib").sha256(p.read_bytes()).hexdigest()
def ink(im):return ImageChops.difference(im.convert('RGB'),Image.new('RGB',im.size,'white')).getbbox()
def trim(source,target,number,padding_pt=2):
 source=Path(source);target=Path(target);target.mkdir(parents=True,exist_ok=True);stem=f'Figure{number}'
 png=Image.open(source/(stem+'.png'));old_pdf=fitz.open(source/(stem+'.pdf'));page=old_pdf[0];W,H=page.rect.width,page.rect.height
 bbox=ink(png);assert bbox
 top=bbox[1]/png.height*H;bottom=bbox[3]/png.height*H
 textboxes=[s['bbox'] for b in page.get_text('dict')['blocks'] if 'lines' in b for line in b['lines'] for s in line['spans']]
 if textboxes:top=min(top,min(b[1] for b in textboxes));bottom=max(bottom,max(b[3] for b in textboxes))
 top=max(0,top-padding_pt);bottom=min(H,bottom+padding_pt)
 report=dict(figure=number,old_canvas_pt=[W,H],new_canvas_pt=[W,bottom-top],crop_top_pt=top,crop_bottom_pt=H-bottom,padding_pt=padding_pt,physical_width_unchanged=True,panels_only_translated=True,formats={})
 for ext in ['png','tiff']:
  path=source/(stem+'.'+ext);im=Image.open(path);box=(0,math.floor(top/H*im.height),im.width,math.ceil(bottom/H*im.height));out=im.crop(box)
  assert im.crop((0,0,im.width,box[1])).convert('RGB').getextrema()==((255,255),)*3
  assert im.crop((0,box[3],im.width,im.height)).convert('RGB').getextrema()==((255,255),)*3
  assert out.tobytes()==im.crop(box).tobytes()
  opts={'dpi':im.info.get('dpi',(300,300) if ext=='png' else (600,600))}
  if ext=='tiff':opts['compression']='tiff_lzw'
  out.save(target/(stem+'.'+ext),**opts)
  report['formats'][ext]=dict(before_sha256=sha(path),after_sha256=sha(target/(stem+'.'+ext)),before_pixels=list(im.size),after_pixels=list(out.size),crop_box=list(box),retained_pixels_identical=True,removed_pixels_all_white=True)
 # A PDF crop changes its visible canvas only; streams, fonts and data marks are retained.
 before_text=page.get_text();before_drawings=len(page.get_drawings())
 page.set_cropbox(fitz.Rect(0,top,W,bottom));old_pdf.save(target/(stem+'.pdf'),garbage=0,deflate=False)
 new_pdf=fitz.open(target/(stem+'.pdf'));assert new_pdf[0].get_text()==before_text;assert len(new_pdf[0].get_drawings())==before_drawings
 report['formats']['pdf']=dict(before_sha256=sha(source/(stem+'.pdf')),after_sha256=sha(target/(stem+'.pdf')),text_identical=True,vector_drawing_count_unchanged=True)
 p=source/(stem+'.svg');tree=etree.parse(str(p));root=tree.getroot();view=list(map(float,root.get('viewBox').split()));assert abs(view[2]-W)<.01 and abs(view[3]-H)<.01
 # Keep all child elements, clip paths, text positions and SVG assets unchanged.
 children=[etree.tostring(x) for x in root];root.set('height',f'{bottom-top:.9f}pt');root.set('viewBox',f'0 {top:.9f} {W:.9f} {bottom-top:.9f}');assert children==[etree.tostring(x) for x in root]
 tree.write(str(target/(stem+'.svg')),encoding='utf-8',xml_declaration=True)
 report['formats']['svg']=dict(before_sha256=sha(p),after_sha256=sha(target/(stem+'.svg')),all_child_elements_identical=True)
 return report

final=ROOT/'output/Main_Figures';reports=[]
for n in range(1,6):
    report=trim(O,final,n);reports.append(report)
    for ext in ['pdf','svg','png','tiff']:
        (final/f'Figure{n}.{ext}').replace(final/f'Figure_{n}.{ext}')
(Q/'trim_report.json').write_text(json.dumps(reports,indent=2),encoding='utf-8')
(Q/'source_data_manifest.json').write_text(json.dumps({n:dict(rows=len(f),columns=list(f)) for n,f in TABLES.items()},indent=2),encoding='utf-8')
print('Five current main figures exported. Frozen source CSVs were not modified.')
